In [4]:
import requests
import pandas as pd
import time
import re
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

In [5]:
class RussianOllamaClient:
    def __init__(self, model_name="llama3:latest"):
        self.model_name = model_name
        self.base_url = "http://localhost:11434"
        print(f"Используем модель: {model_name}")
    
    def make_request(self, prompt: str, max_tokens: int = 100) -> str:
        data = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.3,
                "num_predict": max_tokens
            }
        }
        
        try:
            response = requests.post(
                f"{self.base_url}/api/generate", 
                json=data, 
                timeout=500
            )
            
            if response.status_code == 200:
                result = response.json()
                return result['response']
            else:
                print(f"❌ Ошибка {response.status_code}")
                return ""
                
        except Exception as e:
            print(f"❌ Исключение: {e}")
            return ""

In [6]:
# ollama serve
# & "C:\Users\zvrva\AppData\Local\Programs\Ollama\ollama.exe" serve

In [7]:
client = RussianOllamaClient("llama3:latest")  
    
test_response = client.make_request("Скажи 'тест пройден'")
if test_response:
    print("✅ Модель работает")
else:
    print("❌ Модель не отвечает")

Используем модель: llama3:latest
✅ Модель работает


In [8]:
try:
    df = pd.read_csv('ru_cefr_short.csv')
    print("✅ Файл найден:")
except:
    print("❌ Файл не найден, используем тестовые данные:")
    test_data = {
        'fragment': [
            "Весной, летом и осенью почти каждую субботу он играет в футбол. У них в школе есть футбольная команда. А зимой он играет в хоккей. Ещё мы любим театр.",
            "Вчера я весь вечер повторял грамматику, учил слова. В контрольной работе я сделал 3 ошибки. Питер и Кен написали всё без ошибок. Преподаватель сказал, что они делают большие успехи.",
            "В такой ситуации особенно сложно работающим студентам, которые должны находить время и для учёбы, и для работы. Это нередко вызывает стресс.",
            "Как редкое животное они стоили очень дорого и являлись предметом роскоши. Археологи нашли останки этих животных в развалинах домов богатых людей четвёртого века нашей эры.",
            "Он увлёкся энтомологией ещё мальчиком и в детстве познакомился с основными учёными трудами в этой области.",
            "Так же радиация — это частица, которая летит на огромной скорости. Сама частица может быть почти любой: от атомного ядра до электрона или фотона."
        ],
        'textbook-assigned cefr level': [1, 2, 3, 4, 5, 6]
    }
    df = pd.DataFrame(test_data)

df

✅ Файл найден:


,fragment,textbook-assigned cefr level
0,"Весной, летом и осенью почти каждую субботу он...",1
1,"Все говорят, что мама хорошая хозяйка. А ещё н...",1
2,На каждой двери красные плакаты и красные фона...,1
3,"Я считаю деньги, в час обедаю в кафе, а потом ...",1
4,Магазин «Чёрный квадрат» открывается в 9 часов...,1
...,...,...
7317,Утечка мозгов стала ключевым трендом междунаро...,6
7318,"По оценкам менеджеров «Промы», такая ситуация ...",6
7319,"Но это не мы, а техно-мемы заполоняют мир благ...",6
7320,Mapillary использует программное обеспечение д...,6


In [9]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['fragment'].values,
    df['textbook-assigned cefr level'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['textbook-assigned cefr level']
)

len(train_texts)

5857

In [10]:
texts_to_augment = []

for i in range(len(train_labels)):
  if train_labels[i] == 6:
    texts_to_augment.append(train_texts[i])


len(texts_to_augment)

120

# 3. Изменение порядка предложений

In [8]:
def create_prompt(text):
    return f'''Перепиши этот текст, изменив порядок предложений или частей предложений, сохраняя исходный смысл и структуру. Текст должен оставаться на том же уровне сложности.

Пример:

Оригинал: "Исследователи работают над системами, способными анализировать дорогу лучше, чем водитель-человек."

Аугментация: "Системы, способные анализировать дорогу лучше, чем человек, сейчас разрабатывают исследователи."
    
Текст: "{text}"

Верни ТОЛЬКО новый текст на русском языке без пояснений и комментариев, не заключай новый текст в кавычки:'''

In [9]:
df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	Сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов стерилизуют фрукты и овощи, а также лечат людей. Собственно, это происходит в периоде 3:38–3:54.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Waymo уже запустила приложение на Android для владельцев их самоуправляемых машин, а компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из подполья» и дру

In [10]:
df_pred.to_csv("c2_augmented_3.csv")

# 4. Реорганизация фраз

In [11]:
def create_prompt(text):
    return f'''Перепиши этот текст, изменив порядок слов в предложении, сохраняя исходный смысл и структуру. Текст должен оставаться на том же уровне сложности.

Пример:

Оригинал: "Бедный, так и не выпьешь нормально. Не расслабишься."


Аугментация: "Не расслабишься, бедный, не выпьешь нормально."
    
Текст: "{text}"

Верни ТОЛЬКО новый текст на русском языке без пояснений и комментариев, не заключай новый текст в кавычки:'''

In [12]:
df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	Сама по себе радиация незаразна, собственно. Фрукты и овощи, а также людей часто стерилизуют мощными дозами излучения от тех же самых изотопов, например, 3:38–3:54.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин, а компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения недавно.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из подполья» и другие, рассказы, п

In [13]:
df_pred.to_csv("c2_augmented_4.csv")

# 5. Усложнение объяснений

In [14]:
def create_prompt(text):
    return f'''Перепиши этот текст, сделав объяснения более детальными и развернутыми. Добавь дополнительные пояснения, уточнения, примеры или расширенные определения ключевых понятий.
Пример:

Оригинал: "В мире обострилась конкуренция за таланты: их переманивают высокими зарплатами..."

Аугментация: "В мире возникла жесткая борьба за таланты. Люди меняют работу ради высоких зарплат и перспектив..."

Текст: "{text}"

Верни ТОЛЬКО новый текст на русском языке без пояснений и комментариев, не заключай новый текст в кавычки:'''

In [15]:
df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	Сама по себе радиация не является заражающей. Это означает, что радиация не может передаваться от человека к человеку или от объекта к объекту, как это происходит с инфекциями. В отличие от инфекций, которые могут распространяться через контакт с зараженным человеком или объектом, радиация не имеет способа передачи от одного места к другому.

Например, мощные дозы излучения,

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	В последнее время наблюдается бурное развитие технологий в области автоматизированного вождения, что привело к масси

In [16]:
df_pred.to_csv("c2_augmented_5.csv")

# 8. Использование аналогий

In [17]:
def create_prompt(text):
    return f'''Перепиши следующий текст, используя аналогию для объяснения основного понятия. Используй такую аналогию, которая сделает текст более наглядным, но при этом сохранит высокий уровень сложности и не упростит содержание. Аналогия должна быть метафорической и соответствовать интеллектуальному уровню оригинала. Важно, чтобы аналогия не заменила ключевое содержание текста, а лишь помогла лучше донести его суть.

Пример:

Оригинал: "В мире обострилась конкуренция за таланты..."

Аугментация: "Конкуренция за таланты теперь как гонка за редким ресурсом, который необходим каждому высокотехнологичному бизнесу."

Текст: "{text}"

Верни ТОЛЬКО новый текст на русском языке без пояснений и комментариев, не заключай новый текст в кавычки:'''

In [18]:
df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	Сама по себе радиация, как мощный химикат, который может быть использован для создания уникальных сочетаний свойств, стерилизуя фрукты и овощи, или для лечения людей, как если бы это был сложный синтез лекарственного препарата.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	В мире технологий, где искусственный интеллект играет роль высокопродуктивного фермера, компания Uber, как опытный садовник, вложила один миллиард долларов в собственную систему автоматизированного вождения, чтобы выращивать урожай новых возможностей. А компания Way

In [19]:
df_pred.to_csv("c2_augmented_8.csv")

# 9. Синтаксические трансформации

In [20]:
def create_prompt(text):
    return f'''Перепиши этот текст, заменяя активные грамматические конструкции на пассивные, а пассивные — на активные там, где это возможно и стилистически уместно. Сохраняй исходный смысл, терминологию и общий уровень сложности текста. Обрати внимание на сохранение логических связей между частями предложений.
Пример:

Оригинал: "Множество повестей, рассказов, публицистики, дневников были написаны великим автором."

Аугментация: "Великим автором было написано множество повестей, рассказов, публицистики и дневников."

Текст: "{text}"

Верни ТОЛЬКО новый текст на русском языке без пояснений и комментариев, не заключай новый текст в кавычки:'''

In [21]:
df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	Великие дозы излучения от тех же самых изотопов стерилизуют фрукты и овощи мощными дозами излучения, а также лечат людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Великим компаниям Uber и Waymo было объявлено об инвестиции одного миллиарда долларов в собственные системы автоматизированного вождения, а компания Waymo активировала приложение на Android для владельцев своих самоуправляемых машин.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из подполья» и другие, рассказы, публицистика, дневники, даже

In [22]:
df_pred.to_csv("c2_augmented_9.csv")

# 11. Перефразирование

In [23]:
def create_prompt(text):
    return f"""Перефразируй этот текст на русском языке, сохраняя его уровень сложности. Текст должен оставаться на том же уровне сложности.

Пример:

Оригинал: 'Искусственный интеллект меняет способы взаимодействия людей с технологиями.'

Аугментация: 'Технологии искусственного интеллекта трансформируют методы коммуникации человека с цифровыми устройствами.'

Текст: '{text}'

Верни ТОЛЬКО новый текст на русском языке без пояснений и комментариев, не заключай новый текст в кавычки:"""

In [24]:
df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	Самопо себе радиация не заразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также используются для лечения людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о вложении в миллиард долларов в собственную систему автоматизированного управления, а компания Waymo уже запустила приложение для владельцев ее самоуправляемых автомобилей на Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из подполья» и другие, рассказы, публицист

In [25]:
df_pred.to_csv("c2_augmented_11.csv")

# Генерация по требованиям + пример

In [11]:
def create_prompt():
    return f"""
Сгенерируй  текст на русском языке уровня сложности С2 длиной от 15 до 30 слов.

Требования к уровню С2:
Владение сложной лексикой, в том числе научного стиля речи, устойчивыми оборотами речи, идиомами. Синтаксические модели выражения логической связи между объектами мысли, употребление вводных слов, конструкций научного стиля речи: что является чем, представляет собой, относится к, предназначено для, обозначается как, ведет к чему, состоит из чего, из чего следует что, характеризуется чем, из чего можно сделать вывод и пр. Уровень С2 подразумевает владение языком на профессиональном уровне, включающем преподавание русского языка как иностранного и проведение научно-исследовательской работы.

Пример текста уровня С2: 
Покачиваясь вместе с вагоном, мы преодолевали бескрайние просторы Сибири, и, чем дальше мы уезжали от столиц и больших городов, тем чаще ловили себя на мысли, что сама протяжённость этого пути, постоянная смена пейзажа и наших попутчиков, становится самоцелью путешествия, расстояния измеряются здесь уже не километрами, а прожитыми мгновениями: как пелось в той песне, “есть только миг”. 

Комментарий: очень сложная структура предложения с множеством связей, сложные союзы и предлоги (чем, тем), устойчивые выражения (бескрайние просторы, ловили себя на мысли), связь через двоеточие, причастные и деепричастные обороты, перечисления, использование прецедентных текстов (“есть только миг”.)

Верни только новый текст на русском языке без пояснений и комментариев, верни только ОДИН новый текст, не заключай новый текст в кавычки:"""

In [12]:
df_pred = pd.DataFrame(columns=['generated-text'])

print(f"Генерируем {len(texts_to_augment)} текстов...")

for i in range(len(texts_to_augment)):

    prompt = create_prompt()
    generated_text = client.make_request(prompt)
    
    print(f"{i}. Сгенерированный текст:\n\t{generated_text}")
   
    new_row = pd.DataFrame({
        'generated-text': [generated_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)



Генерируем 120 текстов...
0. Сгенерированный текст:
	Функциональная структура современного общества характеризуется постоянной эволюцией и адаптацией к изменяющимся условиям жизни. Ведущие роли в этом процессе играют технологии, которые представляют собой инструментарий для решения конкретных задач и достижения определенных целей. Системы искусственного интеллекта обозначаются как ключевые факторы развития экономики и социальной сферы
1. Сгенерированный текст:
	Система искусственного интеллекта, предназначенная для анализа больших объемов данных, характеризуется высокой степенью автоматизации и гибкостью. Она может обрабатывать различные типы информации, включая тексты, изображения и звук, что позволяет использовать ее в различных областях, таких как машинное зрение, естественный язык и обработка знаний.
2. Сгенерированный текст:
	Современная информационная инфраструктура обозначается как основа функционирования современного общества, ведущая к формированию глобальной цивилизации. Она 

In [13]:
df_pred.to_csv("c2_generated_with_example_llama3.csv")

# Генерация по требованиям + без примеров

In [14]:
def create_prompt():
    return f"""
Сгенерируй  текст на русском языке уровня сложности С2 длиной от 15 до 30 слов.

Требования к уровню С2:
Владение сложной лексикой, в том числе научного стиля речи, устойчивыми оборотами речи, идиомами. Синтаксические модели выражения логической связи между объектами мысли, употребление вводных слов, конструкций научного стиля речи: что является чем, представляет собой, относится к, предназначено для, обозначается как, ведет к чему, состоит из чего, из чего следует что, характеризуется чем, из чего можно сделать вывод и пр. Уровень С2 подразумевает владение языком на профессиональном уровне, включающем преподавание русского языка как иностранного и проведение научно-исследовательской работы. Очень сложная структура предложения с множеством связей, сложные союзы и предлоги, устойчивые выражения, связь через двоеточие, причастные и деепричастные обороты, перечисления, использование прецедентных текстов.

Верни только новый текст на русском языке без пояснений и комментариев, верни только ОДИН новый текст, не заключай новый текст в кавычки:"""

In [16]:
df_pred = pd.DataFrame(columns=['generated-text'])

print(f"Генерируем {len(texts_to_augment)} текстов...")

for i in range(len(texts_to_augment)):

    prompt = create_prompt()
    generated_text = client.make_request(prompt)
    
    print(f"{i}Сгенерированный текст:\n\t{generated_text}")
   
    new_row = pd.DataFrame({
        'generated-text': [generated_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

df_pred.to_csv("c2_generated_llama3.csv")

Генерируем 120 текстов...
0Сгенерированный текст:
	Связь между концепциями искусственного интеллекта и машинного обучения обозначается как взаимосвязь, ведущая к созданию сложных алгоритмов. Теоретические основы этих дисциплин представляют собой совокупность математических моделей и статистических методов, которые характеризуются высокой степенью абстракции и сложности. В свою очередь, эти модели обрабатываются
1Сгенерированный текст:
	Связь между индивидуальными характеристиками человека и его поведением обозначается как фундаментальная проблема психологии. Она ведет к пониманию природы человеческой личности, представляющей собой сложный системный процесс, который состоит из взаимодействующих компонентов: наследственности, окружающей среды и личных опытов. Характеризуется чем-то новым - эмоциональной рег
2Сгенерированный текст:
	Модель машинного обучения, основанная на глубоких нейронных сетях, предназначена для анализа и интерпретации больших объемов данных. Она характеризуется высок

# Аугментация В2

In [17]:
texts_to_augment_b2 = []

for i in range(len(train_labels)):
  if train_labels[i] == 4:
    texts_to_augment_b2.append(train_texts[i])

texts_to_augment_b2 = texts_to_augment_b2[:120]
len(texts_to_augment_b2)

120

In [18]:

def create_prompt(text):
    return f'''Перепиши данный текст на русском языке так, чтобы его уровень сложности соответствовал С2, сохранив исходный смысл, ключевую информацию и общую тему.

Требования к результату:
- Новый текст должен быть написан на русском языке.
- Уровень сложности должен соответствовать С2.
- Не упрощай текст.
- Сохрани основной смысл исходного текста.
- Не меняй тему текста.
- Не добавляй новые факты, которых нет в исходном тексте.
- Итоговый текст должен звучать естественно и связно.
- Длина нового текста может быть больше исходного, если это необходимо для достижения уровня С2.

Требования к уровню С2:
- Владение сложной лексикой, в том числе научного стиля речи, устойчивыми оборотами речи, идиомами. 
- Синтаксические модели выражения логической связи между объектами мысли, употребление вводных слов, конструкций научного стиля речи: что является чем, представляет собой, относится к, предназначено для, обозначается как, ведет к чему, состоит из чего, из чего следует что, характеризуется чем, из чего можно сделать вывод и пр. 
- Владение языком на профессиональном уровне, включающем преподавание русского языка как иностранного и проведение научно-исследовательской работы. 
- Очень сложная структура предложения с множеством связей, сложные союзы и предлоги, устойчивые выражения, связь через двоеточие, причастные и деепричастные обороты, перечисления, использование прецедентных текстов.

Исходный текст: "{text}"

Верни ТОЛЬКО один новый текст на русском языке без пояснений, комментариев и кавычек:'''

In [19]:
df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment_b2)} текстов...")

for text in texts_to_augment_b2:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1

df_pred.to_csv("c2_from_b2_augmented_llama3.csv")

Аугментируем 120 текстов...

1. Реальный текст:
	По дороге к её дому его ожидали разные препятствия: друзья и подруги невесты не хотели его пропускать, и он должен был что-нибудь подарить или заплатить им.
Аугметированный текст:
	На пути к ее дому его ожидали различные барьеры: знакомые и подруги невесты не хотели позволить ему проходить мимо, и он был вынужден предпринять какое-то действие или заплатить им за проезд.

2. Реальный текст:
	Что же выявил наш анализ? Самое главное — длительность жизни людей творческих профессий была значительно больше, чем средняя, во все эпохи.
Аугметированный текст:
	Анализ выяснил следующее: наиболее значимым результатом является существенная разница в продолжительности жизни людей, занимающихся творческими профессиями, по сравнению с общей средней продолжительностью жизни, которая сохранялась на протяжении всех эпох.

3. Реальный текст:
	В начале конференции немецкий социолог сделал небольшое сообщение о проблемах экологического самосознания в ФРГ, чт

# Аугментация С1

In [20]:
texts_to_augment_c1 = []

for i in range(len(train_labels)):
  if train_labels[i] == 5:
    texts_to_augment_c1.append(train_texts[i])

texts_to_augment_c1 = texts_to_augment_c1[:120]
len(texts_to_augment_c1)

120

In [21]:

def create_prompt(text):
    return f'''Перепиши данный текст на русском языке так, чтобы его уровень сложности соответствовал С2, сохранив исходный смысл, ключевую информацию и общую тему.

Требования к результату:
- Новый текст должен быть написан на русском языке.
- Уровень сложности должен соответствовать С2.
- Не упрощай текст.
- Сохрани основной смысл исходного текста.
- Не меняй тему текста.
- Не добавляй новые факты, которых нет в исходном тексте.
- Итоговый текст должен звучать естественно и связно.
- Длина нового текста может быть больше исходного, если это необходимо для достижения уровня С2.

Требования к уровню С2:
- Владение сложной лексикой, в том числе научного стиля речи, устойчивыми оборотами речи, идиомами. 
- Синтаксические модели выражения логической связи между объектами мысли, употребление вводных слов, конструкций научного стиля речи: что является чем, представляет собой, относится к, предназначено для, обозначается как, ведет к чему, состоит из чего, из чего следует что, характеризуется чем, из чего можно сделать вывод и пр. 
- Владение языком на профессиональном уровне, включающем преподавание русского языка как иностранного и проведение научно-исследовательской работы. 
- Очень сложная структура предложения с множеством связей, сложные союзы и предлоги, устойчивые выражения, связь через двоеточие, причастные и деепричастные обороты, перечисления, использование прецедентных текстов.

Исходный текст: "{text}"

Верни ТОЛЬКО один новый текст на русском языке без пояснений, комментариев и кавычек:'''

In [22]:
df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment_c1)} текстов...")

for text in texts_to_augment_c1:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


df_pred.to_csv("c2_from_c1_augmented_llama3.csv")

Аугментируем 120 текстов...

1. Реальный текст:
	И я понял: это моя жена. На бледном лице сияют глаза. Большие, выпуклые, чёрные! Это мои глаза, моя душа.
Аугметированный текст:
	Я пришел к выводу, что это моя жена. На ее лице сияют глаза, которые напоминают мои собственные - большие, выпуклые, чёрные. Это не только физическое сходство, но и глубокое понимание, которое связывает нас на уровне души.

В этом контексте важна роль эмоциональной привязанности, которая позволяет нам чувствовать себя комфортно в обществе другого

2. Реальный текст:
	Получается безвыходная ситуация: человек, обладающий свободным временем и возможностью полностью поменять свою жизнь, замыкается в себе.
Аугметированный текст:
	Получается сложная дилемма: индивидуум, располагающий свободным временем и возможностью кардинально изменить свою жизнь, оказывается в ловушке собственных ограничений.

3. Реальный текст:
	Почитание святого Георгия, возникшее, вероятно, на территории Каппадокии в V—VI веках, к IX-XI векам 